# 🧠 Quiz Generator — Test Notebook
Test tính năng sinh câu hỏi MCQ từ tài liệu RAG

In [1]:
# Cell 1: Setup path
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # Trỏ về root project
print("✅ Path OK")

✅ Path OK


In [2]:
# Cell 2: Import
from backend.rag.retriever import retrive_context
from backend.rag.generator import get_llm
from backend.models.schemas import QuizResponse, QuizQuestion
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
print("✅ Import OK")

✅ Import OK


In [3]:
# Cell 3: Cấu hình — SỬA Ở ĐÂY
TOPIC       = "Tiêu chí đánh giá môn học"
N_QUESTIONS = 10
DIFFICULTY  = "medium"    # easy | medium | hard

print(f"Topic     : {TOPIC}")
print(f"Số câu    : {N_QUESTIONS}")
print(f"Độ khó    : {DIFFICULTY}")

Topic     : Tiêu chí đánh giá môn học
Số câu    : 10
Độ khó    : medium


In [4]:
# Cell 4: RAG — Tìm tài liệu
chunks = retrive_context(TOPIC, k=5)
print(f"📚 Tìm thấy {len(chunks)} chunks từ knowledge base")

# Xem thử nội dung chunk đầu tiên
if chunks:
    print("\n--- Chunk mẫu (đầu tiên) ---")
    print(chunks[0]['text'][:300], "...")
else:
    print("❌ Không có tài liệu! Upload PDF trước.")

Đang tải mô hình từ HuggingFace về
Đang kết nối Chroma DB ở d:\2025-2026 HKII\multimodel_e_learning\data\chroma_db


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


📚 Tìm thấy 5 chunks từ knowledge base

--- Chunk mẫu (đầu tiên) ---
Nhập môn học máy
ĐồÁn 3
4.2.1. Tiêu chí 1: Đọc hiểu tài liệu (20 điểm)
Được đánh giá qua phần hỏi đáp trong buổi thuyết trình.
Mức độ
Điểm
Mô tả
Xuất sắc
18–20
Tất cảthành viên trảlời trôi chảy, chính xác, có
chiều sâu; liên hệđược với kiến thức bên ngoài
tutorial.
Tốt
14–17
Phần lớn câu hỏi được tr ...


In [5]:
# Cell 5: AI sinh câu hỏi MCQ
DIFFICULTY_MAP = {
    "easy":   "đơn giản, chỉ cần nhớ định nghĩa/khái niệm cơ bản",
    "medium": "trung bình, cần hiểu và áp dụng kiến thức",
    "hard":   "khó, cần phân tích, so sánh, hoặc suy luận sâu",
}

context_text = "\n".join([c["text"] for c in chunks])
llm    = get_llm()
parser = JsonOutputParser(pydantic_object=QuizResponse)

prompt = PromptTemplate(
    template="""
Bạn là giảng viên đại học. Dựa vào TÀI LIỆU sau, hãy tạo {n_questions} câu hỏi MCQ
về chủ đề: "{topic}". Độ khó: {difficulty_desc}

TÀI LIỆU:
{context}

YÊU CẦU:
- Mỗi câu có đúng 4 lựa chọn (list): A, B, C, D
- correct_answer chỉ chứa 1 ký tự: "A", "B", "C" hoặc "D"
- explanation giải thích ngắn tại sao đúng

{format_instructions}
""",
    input_variables=["topic", "n_questions", "difficulty_desc", "context"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain  = prompt | llm | parser
print("⏳ AI đang sinh câu hỏi...")
result = chain.invoke({
    "topic": TOPIC,
    "n_questions": N_QUESTIONS,
    "difficulty_desc": DIFFICULTY_MAP[DIFFICULTY],
    "context": context_text,
})
print("✅ Xong!")

⏳ AI đang sinh câu hỏi...
✅ Xong!


In [6]:
# Cell 6: Hiển thị kết quả đẹp bằng HTML
from IPython.display import display, HTML

questions = result.get("questions", []) if isinstance(result, dict) else result.questions

html = f"""
<style>
  .quiz-card {{ background:#f8f9ff; border-left:4px solid #4F8EF7;
               border-radius:8px; padding:16px 20px; margin:12px 0; }}
  .q-num     {{ color:#4F8EF7; font-weight:bold; font-size:13px; margin-bottom:6px; }}
  .q-text    {{ font-size:16px; font-weight:600; margin-bottom:12px; }}
  .choice    {{ padding:6px 12px; border-radius:6px; margin:4px 0;
               background:#fff; border:1px solid #dde; font-size:14px; }}
  .correct   {{ background:#d4edda; border-color:#28a745; font-weight:bold; }}
  .explain   {{ margin-top:10px; padding:8px 12px; background:#fffde7;
               border-left:3px solid #ffc107; border-radius:4px;
               font-size:13px; color:#555; }}
</style>
<h2 style='color:#1a2c6b'>📝 Quiz: {TOPIC}</h2>
"""

for i, q in enumerate(questions, 1):
    q_data = q if isinstance(q, dict) else q.dict()
    correct = q_data["correct_answer"]

    choices_html = ""
    for j, choice_text in enumerate(q_data["choices"]):
        letter = chr(65 + j)
        css    = "choice correct" if letter == correct else "choice"
        mark   = " ✅" if letter == correct else ""
        choices_html += f'<div class="{css}">{letter}. {choice_text}{mark}</div>'

    html += f"""
    <div class="quiz-card">
      <div class="q-num">Câu {i}</div>
      <div class="q-text">{q_data['question']}</div>
      {choices_html}
      <div class="explain">💡 {q_data['explanation']}</div>
    </div>
    """

html += f"<p style='color:#888'>Tổng: {len(questions)} câu hỏi</p>"
display(HTML(html))